# Urban Area Analysis

> Production runs should use the thin package call in the configuration/import cells below. The reusable `urban_area` package now separates the morphological built-up urban area from PT service/catchment filtering logic.

This notebook calculates precise urban areas using population density, building footprints, and road network data. The analysis uses a sophisticated region-growing algorithm with edge refinement to identify contiguous urban areas with sub-hexagon precision.

## Methodology Overview

### 1. Data Loading & Preparation

- **City Boundary**: Load administrative boundaries from geopackage
- **Population Data**: Load hexagonal population density grids
- **Building Footprints**: Query or load cached building data from OpenStreetMap
- **Road Networks**: Query or load cached road network data from OpenStreetMap
- **Spatial Alignment**: Ensure all datasets use consistent coordinate reference system

### 2. Spatial Data Aggregation

- **Building Analysis**: Aggregate building footprints to population hexagons (area and count)
- **Road Analysis**: Aggregate road networks to hexagons (length and density)
- **Metric Calculation**: Calculate normalized urban development scores per hexagon

### 3. Urban Area Detection

- **Region Growing Algorithm**: Start from highest-scoring seed cell and expand to adjacent qualifying cells
- **Multi-criteria Evaluation**: Uses population density, building coverage, and optionally road density
- **Adjustable Strictness**: Three levels (strict/medium/loose) with different thresholds
- **Hole Filling**: Automatically fills gaps surrounded by urban cells
- **Connectivity Assurance**: Ensures final urban area is a single contiguous region

### 4. Edge Refinement (High Precision)

- **Edge Detection**: Identify hexagons on the urban area boundary
- **Sub-cell Creation**: Subdivide edge hexagons into smaller zones (e.g., 4×4 = 16 sub-cells)
- **Granular Analysis**: Calculate building and road metrics for each sub-cell
- **Selective Inclusion**: Only include sub-cells with actual urban development
- **Connectivity Check**: Ensure refined area remains contiguous

### 5. Results & Visualization

- **Interactive Maps**: Population, building, and road density layers with detailed tooltips
- **Statistical Summary**: Comprehensive urban area metrics and comparisons
- **Data Export**: Save results in multiple formats (GeoJSON, GeoPackage) for GIS analysis

## Key Features

- **Development-Focused**: Prioritizes actual signs of urban development (people + buildings)
- **Multi-scale Analysis**: From ~800m hexagons to ~200m sub-cells for precise boundaries
- **Flexible Parameters**: Adjustable thresholds for different urban contexts
- **Quality Assurance**: Guaranteed contiguous results suitable for transport accessibility analysis
- **Performance Optimized**: Efficient algorithms handle large datasets with spatial indexing

## Outputs

- **Urban Cells**: Individual urban elements (hexagons + refined sub-cells)
- **Urban Hull**: Unified boundary polygon of the entire urban area
- **Analysis Dataset**: Complete hexagon grid with all calculated metrics
- **Interactive Visualizations**: Web maps for data exploration and validation


## Configuration


In [1]:
# Thin notebook runner: the reusable urban_area package is the source of truth.
from pathlib import Path

from urban_area import load_city_config, run_urban_area_pipeline

CITY_NAME = "Oslo, Norway"
CONFIG_PATH = Path("city_config.example.json")
QUERY_MISSING = False
WRITE_OUTPUTS = True

config = load_city_config(CONFIG_PATH, city_name=CITY_NAME)
CITY_SHORT_NAME = config.resolved_slug()
DATA_DIR = str(config.data_dir)

print(f"Analyzing morphological built-up area for: {config.city_name}")
print("PT station filtering should use catchment-population logic, not point-in-polygon tests.")

Analyzing morphological built-up area for: Oslo, Norway
PT station filtering should use catchment-population logic, not point-in-polygon tests.


## Imports and Setup


In [2]:
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
Path(DATA_DIR).mkdir(exist_ok=True)

result = run_urban_area_pipeline(config, query_missing=QUERY_MISSING, write_outputs=WRITE_OUTPUTS)

analysis_data = result["analysis_cells"]
urban_cells = result["urban_cells"]
urban_hull = result["urban_hull"]
validation_report = result["validation_report"]
TARGET_CRS = result["metric_crs"]

summary = validation_report["summary"]
print(f"Metric CRS: {TARGET_CRS}")
print(f"Building signal: {result['building_mode']}")
print(f"Urban area: {summary['urban_area_sqkm']:.2f} km²")
print(f"Urban population: {summary['urban_population']:.0f} / {summary['total_population']:.0f}")
print(result["outputs"])


Metric CRS: EPSG:25833
Building signal: footprint
Urban area: 251.10 km²
Urban population: 804251 / 962819
{'analysis_layer': 'data\\population_density.gpkg/population_density_with_buildings_roads_oslo', 'urban_cells_layer': 'data\\urban_areas.gpkg/urban_cells_oslo_growing_refined', 'urban_hull_layer': 'data\\urban_areas.gpkg/urban_area_oslo_growing_refined', 'urban_cells_geojson': 'data\\urban_cells_oslo_growing_refined.geojson', 'provenance_json': 'data\\provenance\\oslo_growing_refined_provenance.json', 'validation_json': 'data\\provenance\\oslo_growing_refined_validation.json'}


## Data Loading Functions


In [3]:
def get_optimal_crs(city_boundary_gdf, city_name=None):
    """
    Automatically determine the optimal projected CRS for a city.

    Strategy:
    1. Check national/regional CRS preferences (lookup table)
    2. Fall back to UTM zone based on city centroid

    Returns:
    - EPSG code as string (e.g., "EPSG:32633")
    """
    import math

    # Lookup table for preferred national/regional CRS
    # These are often more accurate than UTM for specific countries
    national_crs = {
        # Norway
        "Oslo": "EPSG:25833",  # ETRS89 / UTM zone 33N (modern standard for Norway)
        "Bergen": "EPSG:25832",  # ETRS89 / UTM zone 32N
        "Trondheim": "EPSG:25833",  # ETRS89 / UTM zone 33N
        "Stavanger": "EPSG:25832",  # ETRS89 / UTM zone 32N
        # Sweden
        "Stockholm": "EPSG:3006",  # SWEREF99 TM (national grid)
        "Gothenburg": "EPSG:3006",
        "Malmö": "EPSG:3006",
        # Denmark
        "Copenhagen": "EPSG:25832",  # ETRS89 / UTM zone 32N
        # Finland
        "Helsinki": "EPSG:3067",  # ETRS89 / TM35FIN(E,N)
        # UK
        "London": "EPSG:27700",  # OSGB 1936 / British National Grid
        "Manchester": "EPSG:27700",
        "Edinburgh": "EPSG:27700",
        # France
        "Paris": "EPSG:2154",  # RGF93 / Lambert-93
        "Lyon": "EPSG:2154",
        "Marseille": "EPSG:2154",
        # Germany
        "Berlin": "EPSG:25833",  # ETRS89 / UTM zone 33N
        "Munich": "EPSG:25832",  # ETRS89 / UTM zone 32N
        "Hamburg": "EPSG:25832",
        # Netherlands
        "Amsterdam": "EPSG:28992",  # Amersfoort / RD New
        # Spain
        "Madrid": "EPSG:25830",  # ETRS89 / UTM zone 30N
        "Barcelona": "EPSG:25831",  # ETRS89 / UTM zone 31N
        # Italy
        "Rome": "EPSG:32633",  # WGS 84 / UTM zone 33N
        "Milan": "EPSG:32632",  # WGS 84 / UTM zone 32N
        # USA (by city - states have their own state plane systems)
        "New York": "EPSG:32618",  # WGS 84 / UTM zone 18N
        "Los Angeles": "EPSG:32611",  # WGS 84 / UTM zone 11N
        "Chicago": "EPSG:32616",  # WGS 84 / UTM zone 16N
        # Canada
        "Toronto": "EPSG:32617",  # WGS 84 / UTM zone 17N
        "Vancouver": "EPSG:32610",  # WGS 84 / UTM zone 10N
    }

    # Extract city name from "City, Country" format if needed
    if city_name:
        city_short = city_name.split(",")[0].strip()
        if city_short in national_crs:
            print(f"Using preferred CRS for {city_short}: {national_crs[city_short]}")
            return national_crs[city_short]

    # Fall back to UTM zone calculation
    print("Calculating optimal UTM zone from city centroid...")

    # Ensure we're in WGS84 for centroid calculation
    if city_boundary_gdf.crs != "EPSG:4326":
        city_wgs84 = city_boundary_gdf.to_crs("EPSG:4326")
    else:
        city_wgs84 = city_boundary_gdf

    # Get centroid
    centroid = city_wgs84.geometry.centroid.iloc[0]
    lon = centroid.x
    lat = centroid.y

    # Calculate UTM zone from longitude
    # UTM zones are 6 degrees wide, starting at -180°
    utm_zone = int((lon + 180) / 6) + 1

    # Determine hemisphere (North or South)
    hemisphere = "N" if lat >= 0 else "S"

    # EPSG codes for UTM zones:
    # North hemisphere: 32600 + zone (WGS84) or 25800 + zone (ETRS89 for Europe)
    # South hemisphere: 32700 + zone (WGS84)

    # Use ETRS89 for Europe (recommended for most European countries)
    if -10 <= lon <= 40 and 35 <= lat <= 72:  # Rough Europe bounds
        epsg_code = f"EPSG:{25800 + utm_zone}"
        print(f"European location detected - using ETRS89 / UTM zone {utm_zone}N: {epsg_code}")
    else:
        # Use WGS84 UTM for rest of world
        if hemisphere == "N":
            epsg_code = f"EPSG:{32600 + utm_zone}"
        else:
            epsg_code = f"EPSG:{32700 + utm_zone}"
        print(f"Using WGS 84 / UTM zone {utm_zone}{hemisphere}: {epsg_code}")

    return epsg_code


def load_city_boundary(city_name, boundaries_path, target_crs=None):
    """
    Load and reproject city boundary from geopackage.

    If target_crs is None, automatically determines optimal CRS.
    """
    print(f"Loading city boundary for {city_name}...")

    cities = gpd.read_file(boundaries_path, layer="cities_used")
    city_boundary = cities[cities["Name"] == city_name]

    if city_boundary.empty:
        raise ValueError(f"City '{city_name}' not found in boundaries")

    # Auto-detect CRS if not provided
    if target_crs is None:
        target_crs = get_optimal_crs(city_boundary, city_name)

    city_boundary = city_boundary.to_crs(target_crs)
    print(f"City boundary loaded and reprojected to {target_crs}")

    return city_boundary, target_crs


def load_or_query_buildings(city_name, city_short_name, target_crs, city_boundary, data_dir):
    """Load buildings from cache or query from OSM."""
    buildings_path = Path(data_dir) / "building_footprints.gpkg"
    layer_name = f"{city_short_name}_buildings"

    # Check if cached buildings exist
    try:
        layers = fiona.listlayers(str(buildings_path))
        if layer_name in layers:
            print(f"Loading cached buildings for {city_name}...")
            buildings = gpd.read_file(str(buildings_path), layer=layer_name)
            if buildings.crs != target_crs:
                buildings = buildings.to_crs(target_crs)

            # Ensure area column exists
            if "area_sqm" not in buildings.columns:
                buildings["area_sqm"] = buildings.geometry.area

            print(f"Loaded {len(buildings)} cached buildings")
            return buildings
    except fiona.errors.DriverError:
        pass

    # Query from OSM
    print(f"Querying OSM for buildings in {city_name}...")
    query_polygon = city_boundary.to_crs("EPSG:4326")["geometry"].values[0]
    buildings_4326 = ox.features_from_polygon(query_polygon, tags={"building": True})
    buildings = buildings_4326.to_crs(target_crs)
    buildings["area_sqm"] = buildings.geometry.area

    # Cache the results
    try:
        buildings.to_file(str(buildings_path), layer=layer_name, driver="GPKG")
        print(f"Cached buildings to {buildings_path}")
    except Exception as e:
        print(f"Warning: Could not cache buildings: {e}")

    print(f"Queried {len(buildings)} buildings from OSM")
    return buildings


def load_or_query_roads(city_name, city_short_name, target_crs, city_boundary, data_dir):
    """Load roads from cache or query from OSM."""
    roads_path = Path(data_dir) / "road_networks.gpkg"
    layer_name = f"{city_short_name}_roads"

    # Check if cached roads exist
    try:
        layers = fiona.listlayers(str(roads_path))
        if layer_name in layers:
            print(f"Loading cached roads for {city_name}...")
            roads = gpd.read_file(str(roads_path), layer=layer_name)
            if roads.crs != target_crs:
                roads = roads.to_crs(target_crs)

            # Ensure length column exists
            if "length_m" not in roads.columns:
                roads["length_m"] = roads.geometry.length

            print(f"Loaded {len(roads)} cached road segments ({roads['length_m'].sum() / 1000:.1f} km)")
            return roads
    except fiona.errors.DriverError:
        pass

    # Query from OSM
    print(f"Querying OSM for roads in {city_name}...")
    query_polygon = city_boundary.to_crs("EPSG:4326")["geometry"].values[0]

    road_types = {
        "highway": [
            "motorway",
            "trunk",
            "primary",
            "secondary",
            "tertiary",
            "unclassified",
            "residential",
            "motorway_link",
            "trunk_link",
            "primary_link",
            "secondary_link",
            "tertiary_link",
        ]
    }

    roads_4326 = ox.features_from_polygon(query_polygon, tags=road_types)
    roads_4326 = roads_4326[roads_4326.geometry.type == "LineString"]
    roads = roads_4326.to_crs(target_crs)
    roads["length_m"] = roads.geometry.length

    # Prepare for caching
    roads_clean = roads[["geometry", "highway", "name", "length_m"]].copy()
    for col in ["highway", "name"]:
        if col in roads_clean.columns:
            roads_clean[col] = roads_clean[col].astype(str).replace("nan", None)

    # Cache the results
    try:
        roads_clean.to_file(str(roads_path), layer=layer_name, driver="GPKG")
        print(f"Cached roads to {roads_path}")
    except Exception as e:
        print(f"Warning: Could not cache roads: {e}")

    print(f"Queried {len(roads_clean)} road segments ({roads_clean['length_m'].sum() / 1000:.1f} km)")
    return roads_clean


def load_population_data(city_name, data_dir):
    """Load and filter population density data."""
    print(f"Loading population density data for {city_name}...")

    pop_path = Path(data_dir) / "population_density.gpkg"
    boundary_path = Path(data_dir) / "urban_areas.gpkg"

    population_data = gpd.read_file(str(pop_path), layer="population_density_cities_used")
    city_boundaries = gpd.read_file(str(boundary_path), layer="cities_used")

    city_boundary = city_boundaries[city_boundaries["Name"] == city_name]
    if city_boundary.empty:
        raise ValueError(f"Boundary for city '{city_name}' not found")

    # Ensure CRS compatibility
    if population_data.crs != city_boundary.crs:
        population_data = population_data.to_crs(city_boundary.crs)

    # Filter to city area
    pop_filtered = gpd.overlay(population_data, city_boundary, how="intersection")
    pop_filtered["hex_id"] = range(len(pop_filtered))

    print(f"Filtered to {len(pop_filtered)} population hexagons")
    return pop_filtered

## Spatial Analysis Functions


In [4]:
def aggregate_buildings_to_hexagons(buildings, hexagons):
    """
    Aggregate building statistics per hexagon.

    OPTIMIZED: Uses sjoin + targeted intersection instead of full overlay.
    This is significantly faster for large building datasets.
    """
    print("Aggregating building data to hexagons...")

    # Ensure compatible CRS and valid geometries
    if buildings.crs != hexagons.crs:
        buildings = buildings.to_crs(hexagons.crs)

    buildings = buildings[buildings.geom_type.isin(["Polygon", "MultiPolygon"])]
    buildings = buildings[buildings.geometry.is_valid].copy()
    hexagons = hexagons[hexagons.geometry.is_valid].copy()

    if buildings.empty:
        print("No valid buildings found")
        hexagons["total_building_area_sqm"] = 0.0
        hexagons["building_count"] = 0
        return hexagons

    # Add IDs for tracking
    buildings = buildings.reset_index(drop=True)
    buildings["building_id"] = buildings.index
    hexagons = hexagons.reset_index(drop=True)

    # === OPTIMIZATION: Use sjoin instead of overlay ===
    # Step 1: Spatial join to find which buildings intersect which hexagons
    joined = gpd.sjoin(
        buildings[["building_id", "geometry"]], hexagons[["hex_id", "geometry"]], how="inner", predicate="intersects"
    )

    if joined.empty:
        print("No buildings intersect hexagons")
        hexagons["total_building_area_sqm"] = 0.0
        hexagons["building_count"] = 0
        return hexagons

    # Step 2: Calculate intersection areas (only for matched pairs)
    # Group by hex_id for efficient processing
    building_stats = []

    for hex_id, group in joined.groupby("hex_id"):
        hex_geom = hexagons.loc[hexagons["hex_id"] == hex_id, "geometry"].iloc[0]
        total_area = 0.0
        unique_buildings = set()

        for _, row in group.iterrows():
            bldg_geom = buildings.loc[row["building_id"], "geometry"]
            try:
                intersection = bldg_geom.intersection(hex_geom)
                if hasattr(intersection, "area"):
                    total_area += intersection.area
                unique_buildings.add(row["building_id"])
            except Exception:
                continue

        building_stats.append(
            {"hex_id": hex_id, "total_building_area_sqm": total_area, "building_count": len(unique_buildings)}
        )

    stats_df = pd.DataFrame(building_stats)

    # Merge back to hexagons
    result = hexagons.merge(stats_df, on="hex_id", how="left")
    result["total_building_area_sqm"] = result["total_building_area_sqm"].fillna(0)
    result["building_count"] = result["building_count"].fillna(0).astype(int)

    print(f"Aggregated {result['building_count'].sum()} buildings")
    return result


def aggregate_roads_to_hexagons(roads, hexagons):
    """
    Aggregate road statistics per hexagon.

    OPTIMIZED: Uses sjoin + targeted intersection instead of full overlay.
    """
    print("Aggregating road data to hexagons...")

    # Ensure compatible CRS and valid geometries
    if roads.crs != hexagons.crs:
        roads = roads.to_crs(hexagons.crs)

    roads = roads[roads.geometry.is_valid].copy()
    hexagons = hexagons.copy()

    if roads.empty:
        print("No valid roads found")
        hexagons["total_road_length_m"] = 0.0
        hexagons["road_density_km_per_sqkm"] = 0.0
        return hexagons

    # Add IDs for tracking
    roads = roads.reset_index(drop=True)
    roads["road_id"] = roads.index

    # === OPTIMIZATION: Use sjoin instead of overlay ===
    joined = gpd.sjoin(
        roads[["road_id", "geometry"]], hexagons[["hex_id", "geometry"]], how="inner", predicate="intersects"
    )

    if joined.empty:
        print("No roads intersect hexagons")
        hexagons["total_road_length_m"] = 0.0
        hexagons["road_density_km_per_sqkm"] = 0.0
        return hexagons

    # Calculate intersection lengths
    road_stats = []

    for hex_id, group in joined.groupby("hex_id"):
        hex_geom = hexagons.loc[hexagons["hex_id"] == hex_id, "geometry"].iloc[0]
        total_length = 0.0
        unique_roads = set()

        for _, row in group.iterrows():
            road_geom = roads.loc[row["road_id"], "geometry"]
            try:
                intersection = road_geom.intersection(hex_geom)
                if hasattr(intersection, "length"):
                    total_length += intersection.length
                unique_roads.add(row["road_id"])
            except Exception:
                continue

        road_stats.append(
            {"hex_id": hex_id, "total_road_length_m": total_length, "road_segment_count": len(unique_roads)}
        )

    stats_df = pd.DataFrame(road_stats)

    # Merge back and calculate density
    result = hexagons.merge(stats_df, on="hex_id", how="left")
    result["total_road_length_m"] = result["total_road_length_m"].fillna(0)
    result["road_segment_count"] = result["road_segment_count"].fillna(0).astype(int)

    # Calculate road density (km per sq km)
    hex_area_sqm = result.geometry.iloc[0].area
    hex_area_sqkm = hex_area_sqm / 1_000_000
    result["road_density_km_per_sqkm"] = (result["total_road_length_m"] / 1000) / hex_area_sqkm

    total_road_km = result["total_road_length_m"].sum() / 1000
    avg_density = result["road_density_km_per_sqkm"].mean()
    print(f"Aggregated {total_road_km:.1f} km of roads (avg density: {avg_density:.2f} km/sq km)")
    return result

## Load Data


In [5]:
# Load city boundary (with auto-detection of optimal CRS if TARGET_CRS is None)
city_boundary, TARGET_CRS = load_city_boundary(CITY_NAME, BOUNDARIES_PATH, TARGET_CRS)

# Load buildings
city_buildings = load_or_query_buildings(CITY_NAME, CITY_SHORT_NAME, TARGET_CRS, city_boundary, DATA_DIR)

# Load roads
city_roads = load_or_query_roads(CITY_NAME, CITY_SHORT_NAME, TARGET_CRS, city_boundary, DATA_DIR)

# Load population data
population_hexagons = load_population_data(CITY_NAME, DATA_DIR)

print("\nData loading complete:")
print(f"- Buildings: {len(city_buildings):,}")
print(f"- Roads: {len(city_roads):,} segments")
print(f"- Population hexagons: {len(population_hexagons):,}")
print(f"- Using CRS: {TARGET_CRS}")

NameError: name 'BOUNDARIES_PATH' is not defined

## Spatial Analysis


In [ ]:
# Aggregate building data
analysis_data = aggregate_buildings_to_hexagons(city_buildings, population_hexagons)

# Aggregate road data
analysis_data = aggregate_roads_to_hexagons(city_roads, analysis_data)

# Prepare for urban detection
analysis_data = analysis_data.rename(columns={"hex_id": "cell_id"})
analysis_data = analysis_data.to_crs(TARGET_CRS)

# Derived metrics for robust scoring (reduce outlier effects)
analysis_data["hex_area_sqm"] = analysis_data.geometry.area
analysis_data["hex_area_sqkm"] = analysis_data["hex_area_sqm"] / 1_000_000
analysis_data["building_coverage"] = (
    (analysis_data["total_building_area_sqm"] / analysis_data["hex_area_sqm"]).clip(lower=0, upper=1).fillna(0)
)


# Robust min-max normalization between the 5th and 95th percentiles
def _robust_minmax(s: pd.Series, lower: float = 0.05, upper: float = 0.95) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    denom = (hi - lo) if (hi - lo) != 0 else 1.0
    return ((s - lo) / denom).clip(0, 1).fillna(0)


analysis_data["pop_norm"] = _robust_minmax(analysis_data["population"])
analysis_data["bldg_cov_norm"] = _robust_minmax(analysis_data["building_coverage"])
if "road_density_km_per_sqkm" in analysis_data.columns:
    analysis_data["road_norm"] = _robust_minmax(analysis_data["road_density_km_per_sqkm"])

# Base urban score used for seed selection
if "road_norm" in analysis_data.columns:
    analysis_data["urban_score_base"] = (
        0.50 * analysis_data["pop_norm"] + 0.40 * analysis_data["bldg_cov_norm"] + 0.10 * analysis_data["road_norm"]
    )
else:
    analysis_data["urban_score_base"] = 0.55 * analysis_data["pop_norm"] + 0.45 * analysis_data["bldg_cov_norm"]

print(f"\nAnalysis dataset ready: {len(analysis_data)} hexagons with aggregated + derived metrics")

## Data Visualization


In [ ]:
# Create interactive visualization
print("Creating interactive map...")

viz_data = analysis_data.to_crs("EPSG:4326")

# Calculate quantiles for tooltip display
viz_data["pop_quantile"] = (viz_data["population"].rank(pct=True) * 100).round(0).astype(int)
viz_data["building_quantile"] = (viz_data["total_building_area_sqm"].rank(pct=True) * 100).round(0).astype(int)
if "road_density_km_per_sqkm" in viz_data.columns:
    viz_data["road_quantile"] = (viz_data["road_density_km_per_sqkm"].rank(pct=True) * 100).round(0).astype(int)

# Round values for cleaner tooltips
viz_data["population_rounded"] = viz_data["population"].round(0).astype(int)
viz_data["building_count_rounded"] = viz_data["building_count"].astype(int)
viz_data["building_area_rounded"] = viz_data["total_building_area_sqm"].round(0).astype(int)

# Calculate building coverage percentage
hex_area_sqm = analysis_data.geometry.iloc[0].area
viz_data["building_coverage_pct"] = (viz_data["total_building_area_sqm"] / hex_area_sqm * 100).round(0).astype(int)

if "road_density_km_per_sqkm" in viz_data.columns:
    viz_data["road_density_rounded"] = viz_data["road_density_km_per_sqkm"].round(0).astype(int)
    viz_data["road_length_rounded"] = viz_data["total_road_length_m"].round(0).astype(int)

# Prepare tooltip columns with quantiles and rounded values
tooltip_cols = [
    "population_rounded",
    "pop_quantile",
    "building_count_rounded",
    "building_area_rounded",
    "building_quantile",
    "building_coverage_pct",
]

if "road_density_km_per_sqkm" in viz_data.columns:
    tooltip_cols.extend(["road_density_rounded", "road_quantile", "road_length_rounded"])

# Create background layer with all analysis cells
all_cells_viz = analysis_data.to_crs("EPSG:4326")
# Add quantiles and rounded values for tooltips
all_cells_viz["pop_quantile"] = (all_cells_viz["population"].rank(pct=True) * 100).round(0).astype(int)
all_cells_viz["building_quantile"] = (
    (all_cells_viz["total_building_area_sqm"].rank(pct=True) * 100).round(0).astype(int)
)
if "road_density_km_per_sqkm" in all_cells_viz.columns:
    all_cells_viz["road_quantile"] = (
        (all_cells_viz["road_density_km_per_sqkm"].rank(pct=True) * 100).round(0).astype(int)
    )
all_cells_viz["population_display"] = all_cells_viz["population"].round(0).astype(int)
all_cells_viz["building_area_display"] = all_cells_viz["total_building_area_sqm"].round(0).astype(int)
if "road_density_km_per_sqkm" in all_cells_viz.columns:
    all_cells_viz["road_density_display"] = all_cells_viz["road_density_km_per_sqkm"].round(0).astype(int)

# Tooltip for all analysis cells
all_cells_tooltip = ["population_display", "pop_quantile", "building_area_display", "building_quantile"]
if "road_density_km_per_sqkm" in all_cells_viz.columns:
    all_cells_tooltip.extend(["road_density_display", "road_quantile"])

# Create base map with all analysis cells (background layer, hidden by default)
m = all_cells_viz.explore(
    color="lightgrey",
    tooltip=all_cells_tooltip,
    popup=False,
    style_kwds=dict(fillColor="lightgrey", color="darkgrey", fillOpacity=0.5, weight=1, opacity=0.8),
    name="All Analysis Cells",
)

# Add population layer (default)
viz_data.explore(
    m=m,  # Add to existing map
    column="population",
    cmap="viridis",
    tooltip=tooltip_cols,
    popup=True,
    legend=True,
    legend_kwds=dict(caption="Population"),
    style_kwds=dict(fillOpacity=0.7, weight=0.5),
    name="Population Density",
)

# Add building density layer
viz_data.explore(
    m=m,  # Add to existing map
    column="total_building_area_sqm",
    cmap="plasma",
    tooltip=tooltip_cols,
    popup=True,
    legend=True,
    legend_kwds=dict(caption="Building Area (m²)"),
    style_kwds=dict(fillOpacity=0.7, weight=0.5),
    name="Building Density",
)

# Add road density layer if available
if "road_density_km_per_sqkm" in viz_data.columns:
    viz_data.explore(
        m=m,  # Add to existing map
        column="road_density_km_per_sqkm",
        cmap="inferno",
        tooltip=tooltip_cols,
        popup=True,
        legend=True,
        legend_kwds=dict(caption="Road Density (km/sq km)"),
        style_kwds=dict(fillOpacity=0.7, weight=0.5),
        name="Road Density",
    )

# Add layer control to switch between layers
folium.LayerControl().add_to(m)

print(f"Interactive map created with {len(tooltip_cols)} tooltip fields")
print("Layers available:")
print("- Population Density (default)")
print("- Building Density")
if "road_density_km_per_sqkm" in viz_data.columns:
    print("- Road Density")

display(m)

## Urban Area Detection Algorithm


In [ ]:
# Urban area detection algorithm with multi-seed clustering and neighbor-aware growth


def build_adjacency_graph(gdf: gpd.GeoDataFrame, id_col: str = "hex_id") -> dict:
    """
    Pre-compute adjacency graph using spatial index.
    Returns dict mapping each hex_id to set of neighboring hex_ids.

    This is O(n) with spatial index vs O(n²) naive approach.
    """
    from collections import defaultdict

    gdf = gdf.copy().reset_index(drop=True)
    sindex = gdf.sindex
    adjacency = defaultdict(set)

    # Batch query: for each geometry, find candidates via bbox then filter by touches
    for idx, row in gdf.iterrows():
        hid = row[id_col]
        geom = row.geometry
        # Query spatial index for bbox candidates
        candidates_idx = list(sindex.intersection(geom.bounds))
        for cand_idx in candidates_idx:
            if cand_idx == idx:
                continue
            cand_row = gdf.iloc[cand_idx]
            if geom.touches(cand_row.geometry):
                adjacency[hid].add(cand_row[id_col])
                adjacency[cand_row[id_col]].add(hid)  # Symmetric

    return dict(adjacency)


def find_seed_clusters(density, adjacency, seed_quantile=0.90, min_cluster_size=3):
    """
    Find multiple seed clusters from high-scoring cells.
    Returns list of sets, each containing hex_ids for a seed cluster.

    This enables multi-seed growth which better captures polycentric urban areas.
    """
    from collections import deque

    # Get cells above seed threshold
    threshold = density["urban_score_base"].quantile(seed_quantile)
    seed_candidates = set(density[density["urban_score_base"] >= threshold]["hex_id"])

    if not seed_candidates:
        # Fall back to single highest cell
        best_idx = density["urban_score_base"].idxmax()
        return [{density.loc[best_idx, "hex_id"]}]

    # Find connected components among seed candidates
    clusters = []
    visited = set()

    for start_hid in seed_candidates:
        if start_hid in visited:
            continue

        # BFS to find connected cluster
        cluster = set()
        q = deque([start_hid])
        visited.add(start_hid)

        while q:
            hid = q.popleft()
            cluster.add(hid)

            for neighbor in adjacency.get(hid, set()):
                if neighbor in seed_candidates and neighbor not in visited:
                    visited.add(neighbor)
                    q.append(neighbor)

        if len(cluster) >= min_cluster_size:
            clusters.append(cluster)

    # If no clusters meet minimum size, use the largest one or single best cell
    if not clusters:
        if seed_candidates:
            clusters = [seed_candidates]
        else:
            best_idx = density["urban_score_base"].idxmax()
            clusters = [{density.loc[best_idx, "hex_id"]}]

    return clusters


def run_growing_method(
    density,
    urban_strictness: str = "medium",
    water_guard: bool = True,
    multi_seed: bool = True,
    min_seed_cluster_size: int = 2,
):
    """
    Contiguous urban area detection using BFS with multi-seed clustering,
    robust scoring, neighbor-aware rules, and water guardrails.

    MULTI-SEED CLUSTERING:
    - Identifies multiple high-scoring seed clusters
    - Grows from all seeds simultaneously
    - Naturally captures polycentric urban structures
    - Merges overlapping growth areas

    Inputs:
    - density: GeoDataFrame with columns population, total_building_area_sqm, geometry,
               and derived columns pop_norm, bldg_cov_norm, urban_score_base.
    - urban_strictness: one of {"strict", "medium", "loose"}.
    - water_guard: if True, suppresses growth through cells with negligible development.
    - multi_seed: if True, uses multiple seed clusters for growth.
    - min_seed_cluster_size: minimum cells for a seed cluster (ignored if multi_seed=False).

    Output:
    - GeoDataFrame of selected urban cells, contiguous and hole-checked.
    """
    from collections import deque
    import numpy as np

    print(f"Running growing method (strictness: {urban_strictness}, multi_seed: {multi_seed})")

    density = density[density.geometry.is_valid].copy()
    density["hex_id"] = range(len(density))
    density = density.reset_index(drop=True)

    # Parameters per strictness
    params = {
        "strict": {
            "seed_q": 0.92,
            "score_min": 0.40,
            "pop_min": 0.15,
            "bldg_min": 0.12,
            "neighbor_relax": 0.02,
        },
        "medium": {
            "seed_q": 0.88,
            "score_min": 0.30,
            "pop_min": 0.10,
            "bldg_min": 0.08,
            "neighbor_relax": 0.04,
        },
        "loose": {
            "seed_q": 0.80,
            "score_min": 0.24,
            "pop_min": 0.06,
            "bldg_min": 0.05,
            "neighbor_relax": 0.06,
        },
    }[urban_strictness]

    has_road = "road_density_km_per_sqkm" in density.columns

    # === STEP 1: Pre-compute scores BEFORE creating lookup ===
    if has_road and urban_strictness == "strict":
        density["qual_score"] = (
            0.45 * density["pop_norm"] + 0.40 * density["bldg_cov_norm"] + 0.15 * density.get("road_norm", 0)
        )
    else:
        density["qual_score"] = 0.55 * density["pop_norm"] + 0.45 * density["bldg_cov_norm"]

    # === STEP 2: Build adjacency graph ===
    print("Building adjacency graph...")
    adjacency = build_adjacency_graph(density, id_col="hex_id")
    print(f"Adjacency graph built: {len(adjacency)} nodes")

    # === STEP 3: Create lookup AFTER adding qual_score ===
    hex_lookup = density.set_index("hex_id")

    # Pre-compute water guard thresholds
    eps_pop = max(5.0 / (density["population"].max() or 1.0), 0.002)
    eps_bldg = 0.002

    # === STEP 4: Find seed clusters ===
    if multi_seed:
        seed_clusters = find_seed_clusters(
            density, adjacency, seed_quantile=params["seed_q"], min_cluster_size=min_seed_cluster_size
        )
        print(f"Found {len(seed_clusters)} seed cluster(s) with {sum(len(c) for c in seed_clusters)} total seed cells")

        # Initialize with all seed cells
        urban_ids = set()
        for cluster in seed_clusters:
            urban_ids.update(cluster)

        # Initialize BFS queue with all seed cells
        q = deque(urban_ids)
    else:
        # Single seed mode (original behavior)
        seed_idx = density["urban_score_base"].idxmax()
        seed = density.loc[seed_idx]
        seed_hid = seed["hex_id"]
        print(f"Single seed score={seed['urban_score_base']:.3f}")

        urban_ids = {seed_hid}
        q = deque([seed_hid])

    # === STEP 5: BFS growth from all seeds ===
    processed = 0

    while q:
        current_hid = q.popleft()

        # Get neighbors from pre-computed adjacency (O(1) lookup)
        neighbors = adjacency.get(current_hid, set())

        for neighbor_hid in neighbors:
            if neighbor_hid in urban_ids:
                continue

            row = hex_lookup.loc[neighbor_hid]

            # Water guard check
            if water_guard and (row["pop_norm"] < eps_pop and row["bldg_cov_norm"] < eps_bldg):
                continue

            # Calculate neighbor urban share using pre-computed adjacency
            nb_neighbors = adjacency.get(neighbor_hid, set())
            if nb_neighbors:
                nb_urban_share = len(nb_neighbors & urban_ids) / len(nb_neighbors)
            else:
                nb_urban_share = 0.0

            # Relaxation based on neighbor context
            relax = params["neighbor_relax"] * min(1.0, max(0.0, (nb_urban_share - 0.5) * 2))
            score_min = max(0, params["score_min"] - relax)

            # Check qualification
            if (
                row["qual_score"] >= score_min
                and row["pop_norm"] >= params["pop_min"]
                and row["bldg_cov_norm"] >= params["bldg_min"]
            ):
                urban_ids.add(neighbor_hid)
                q.append(neighbor_hid)

        processed += 1
        if processed % 200 == 0:
            print(f"Processed {processed} fronts; size={len(urban_ids)}")

    print(f"Initial growth complete: {len(urban_ids)} cells")

    # === STEP 6: Hole filling using pre-computed adjacency ===
    holes = 0
    for hid in range(len(density)):
        if hid in urban_ids:
            continue
        neighbors = adjacency.get(hid, set())
        if len(neighbors) >= 3 and neighbors.issubset(urban_ids):
            row = hex_lookup.loc[hid]
            if row["pop_norm"] >= eps_pop or row["bldg_cov_norm"] >= eps_bldg:
                urban_ids.add(hid)
                holes += 1
    if holes:
        print(f"Hole filling added {holes} cells")

    if not urban_ids:
        print("No urban area identified")
        return gpd.GeoDataFrame()

    urban_gdf = density[density["hex_id"].isin(urban_ids)].copy()

    # === STEP 7: Keep largest connected component ===
    try:
        from scipy.sparse import csr_matrix
        from scipy.sparse.csgraph import connected_components

        # Build sparse adjacency matrix for urban cells only
        urban_hex_ids = list(urban_gdf["hex_id"])
        hid_to_idx = {hid: i for i, hid in enumerate(urban_hex_ids)}
        n = len(urban_hex_ids)

        rows, cols = [], []
        for hid in urban_hex_ids:
            for neighbor in adjacency.get(hid, set()):
                if neighbor in hid_to_idx:
                    rows.append(hid_to_idx[hid])
                    cols.append(hid_to_idx[neighbor])

        if rows:
            adj_matrix = csr_matrix(([1] * len(rows), (rows, cols)), shape=(n, n))
            n_components, labels = connected_components(adj_matrix, directed=False)
        else:
            n_components, labels = 1, np.zeros(n, dtype=int)

        urban_gdf["component"] = labels

        # Keep largest component by cell count
        keep_label = urban_gdf["component"].value_counts().idxmax()
        result = urban_gdf[urban_gdf["component"] == keep_label].copy()

        print(f"Final contiguous urban area: {len(result)} cells (kept largest of {n_components} components)")

    except ImportError:
        # Fallback to manual method if scipy not available
        print("scipy not available, using manual connected components...")
        urban_gdf["component"] = -1
        vis = set()
        comp_id = 0

        for hid in urban_gdf["hex_id"]:
            if hid in vis:
                continue
            comp_id += 1
            stack = [hid]
            vis.add(hid)
            while stack:
                cid = stack.pop()
                urban_gdf.loc[urban_gdf["hex_id"] == cid, "component"] = comp_id
                for neighbor in adjacency.get(cid, set()):
                    if neighbor not in vis and neighbor in set(urban_gdf["hex_id"]):
                        vis.add(neighbor)
                        stack.append(neighbor)

        keep = urban_gdf["component"].value_counts().idxmax()
        result = urban_gdf[urban_gdf["component"] == keep].copy()
        print(f"Final contiguous urban area: {len(result)} cells (kept largest of {comp_id} components)")

    # Clean up temporary columns
    cols_to_drop = ["component", "urban_score_base", "hex_id", "qual_score"]
    result = result.drop(columns=[c for c in cols_to_drop if c in result.columns], errors="ignore")
    result = result.reset_index(drop=True)

    return result

## Run Urban Area Detection

The urban detection algorithm uses **multi-seed clustering** with **population and building density** as primary indicators:

### Multi-Seed Clustering (New)

Instead of growing from a single seed cell, the algorithm:

1. **Identifies seed clusters**: Finds groups of high-scoring adjacent cells above the seed quantile threshold
2. **Grows simultaneously**: Expands from all seed clusters at once using BFS
3. **Merges naturally**: Overlapping growth areas combine into one contiguous urban area
4. **Better for polycentric cities**: Captures multiple urban cores (e.g., Oslo + surrounding municipalities)

### Strictness Levels

- **"strict"**: Uses all 3 metrics (population, buildings, roads) with high thresholds
  - Uses top 8% as seeds, requires strong performance in most metrics
  - Best for identifying dense urban cores
- **"medium"**: Uses 2 metrics (population, buildings) - **RECOMMENDED**
  - Uses top 12% as seeds, requires both population AND building presence
  - Good balance for developed urban areas
- **"loose"**: Uses 2 metrics with lower thresholds
  - Uses top 20% as seeds, more flexible criteria
  - Captures suburban and lower-density development

### Key Features

- **Multi-Seed Growth**: Multiple starting points for polycentric urban areas
- **Development-Focused**: Prioritizes population + building presence
- **Water Guard**: Prevents growth across water/empty areas
- **Neighbor-Aware Relaxation**: Cells with more urban neighbors qualify more easily
- **Hole Filling**: Automatically fills small gaps surrounded by urban cells
- **Connectivity**: Ensures final urban area is spatially connected


In [ ]:
# Run urban area detection with adjustable strictness and multi-seed clustering
URBAN_STRICTNESS = "medium"  # Adjust to "loose" for more coverage, "strict" for compact cores
USE_MULTI_SEED = True  # Enable multi-seed clustering for polycentric urban areas
MIN_SEED_CLUSTER_SIZE = 2  # Minimum cells to form a seed cluster

# Clear any previous results
urban_cells = None

print(f"Running urban area detection (strictness: {URBAN_STRICTNESS}, multi_seed: {USE_MULTI_SEED})...")
urban_cells = run_growing_method(
    analysis_data.copy(),
    urban_strictness=URBAN_STRICTNESS,
    water_guard=True,
    multi_seed=USE_MULTI_SEED,
    min_seed_cluster_size=MIN_SEED_CLUSTER_SIZE,
)

# Calculate and display summary statistics
if not urban_cells.empty:
    total_pop = analysis_data["population"].sum()
    hex_area_sqm = analysis_data.geometry.iloc[0].area

    urban_pop = urban_cells["population"].sum()
    urban_pop_pct = (urban_pop / total_pop) * 100
    urban_area_pct = (len(urban_cells) / len(analysis_data)) * 100
    avg_road_density = urban_cells.get("road_density_km_per_sqkm", pd.Series(dtype=float)).mean()

    print(f"\n=== URBAN AREA SUMMARY (Strictness: {URBAN_STRICTNESS.upper()}) ===")
    print(f"Urban cells: {len(urban_cells):,} ({urban_area_pct:.1f}% of total cells)")
    print(f"Urban population: {urban_pop:,.0f} ({urban_pop_pct:.1f}% of total)")
    print(f"Urban building area: {urban_cells['total_building_area_sqm'].sum():,.0f} m²")
    if not np.isnan(avg_road_density):
        print(f"Average road density: {avg_road_density:.2f} km/sq km")
else:
    print("No urban cells identified — try 'loose' or review input metrics")

## Visualize Results


## Edge Refinement for Higher Precision

This section refines the urban area boundary by subdividing edge hexagons into smaller zones and only including portions with actual urban development. This creates a more precise boundary that better represents where people live and work.

### Process:

1. **Identify Edge Hexagons**: Find hexagons on the urban area boundary
2. **Create Sub-cells**: Subdivide each edge hexagon into smaller zones (e.g., 4x4 = 16 sub-cells)
3. **Aggregate Metrics**: Calculate building and road density for each sub-cell
4. **Apply Criteria**: Determine which sub-cells qualify as urban
5. **Refine Boundary**: Replace edge hexagons with qualifying sub-cells only


In [ ]:
def refine_urban_edges_optimized(
    urban_cells,
    all_analysis_data,
    buildings,
    roads,
    subdivision_factor=4,
    min_building_area=600,
    min_road_density=3.0,
    require_local_presence=True,
):
    """
    Refine urban area edges by subdividing edge hexagons and analyzing sub-cell development.

    OPTIMIZED VERSION:
    - Uses sjoin instead of overlay for building/road aggregation
    - Batch processes subcell creation
    - Uses vectorized operations for qualification checks
    - Pre-computes adjacency for connectivity check

    Parameters:
    - subdivision_factor: grid per edge hexagon (n x n)
    - min_building_area: m² threshold for sub-cell to qualify
    - min_road_density: km/sq km threshold for sub-cell to qualify
    - require_local_presence: if True, a sub-cell must have either buildings >= 100 m² or estimated population
      share >= 1% of parent to qualify; road density alone is insufficient.
    """
    from shapely.geometry import Polygon
    import numpy as np
    from collections import deque

    print(f"Starting OPTIMIZED edge refinement with {subdivision_factor}x{subdivision_factor} subdivision...")
    print(f"Criteria: >= {min_building_area} m² buildings OR >= {min_road_density} km/sq km roads per sub-cell")
    if require_local_presence:
        print("Plus local presence guard: need >=100 m² buildings or >=1% of parent population")

    if urban_cells.empty:
        print("No urban cells to refine")
        return urban_cells

    # Ensure consistent CRS
    target_crs = urban_cells.crs
    if buildings.crs != target_crs:
        buildings = buildings.to_crs(target_crs)
    if roads.crs != target_crs:
        roads = roads.to_crs(target_crs)

    # Add cell IDs for tracking
    urban_cells = urban_cells.copy()
    all_analysis_data = all_analysis_data.copy()

    if "cell_id" not in urban_cells.columns:
        urban_cells["cell_id"] = range(len(urban_cells))
    if "cell_id" not in all_analysis_data.columns:
        all_analysis_data["cell_id"] = range(len(all_analysis_data))

    # 1. Identify edge hexagons using pre-computed adjacency
    print("\n1. Identifying edge hexagons...")
    urban_ids = set(urban_cells["cell_id"])

    # Build adjacency graph for all analysis cells
    adjacency = build_adjacency_graph(all_analysis_data, id_col="cell_id")

    edge_hexagons = []
    for cell_id in urban_ids:
        neighbors = adjacency.get(cell_id, set())
        # Check if any neighbor is NOT urban
        if any(n not in urban_ids for n in neighbors):
            edge_hexagons.append(cell_id)

    print(f"Found {len(edge_hexagons)} edge hexagons out of {len(urban_cells)} urban cells")
    if not edge_hexagons:
        return urban_cells

    # 2. Create sub-cells in batch
    print(f"\n2. Creating {subdivision_factor}x{subdivision_factor} sub-cells for edge hexagons...")

    def create_subcells_batch(hex_gdf, n):
        """Create subcells for all hexagons in batch."""
        all_subs = []
        all_parents = []

        for _, row in hex_gdf.iterrows():
            hex_geom = row.geometry
            parent_id = row["cell_id"]

            minx, miny, maxx, maxy = hex_geom.bounds
            dx = (maxx - minx) / n
            dy = (maxy - miny) / n

            for i in range(n):
                for j in range(n):
                    rect = Polygon(
                        [
                            (minx + i * dx, miny + j * dy),
                            (minx + (i + 1) * dx, miny + j * dy),
                            (minx + (i + 1) * dx, miny + (j + 1) * dy),
                            (minx + i * dx, miny + (j + 1) * dy),
                        ]
                    )
                    if hex_geom.contains(rect):
                        all_subs.append(rect)
                        all_parents.append(parent_id)
                    else:
                        inter = rect.intersection(hex_geom)
                        if hasattr(inter, "area") and inter.area > 0:
                            all_subs.append(inter)
                            all_parents.append(parent_id)

        return all_subs, all_parents

    edge_cells = urban_cells[urban_cells["cell_id"].isin(edge_hexagons)]
    subs, parents = create_subcells_batch(edge_cells, subdivision_factor)

    subcells_gdf = gpd.GeoDataFrame(
        {"geometry": subs, "parent_hex_id": parents, "subcell_id": range(len(subs))}, crs=target_crs
    )

    if subcells_gdf.empty:
        return urban_cells

    print(f"Created {len(subcells_gdf)} sub-cells")

    # 3. Aggregate buildings and roads using sjoin
    print("\n3. Aggregating buildings and roads to sub-cells...")

    subcells_gdf["building_area_sqm"] = 0.0
    subcells_gdf["road_length_m"] = 0.0

    # Buildings aggregation with sjoin
    if not buildings.empty:
        b = buildings[buildings.geometry.is_valid & buildings.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
        if not b.empty:
            b = b.reset_index(drop=True)
            b["bid"] = b.index

            # Spatial join to find intersecting pairs
            joined = gpd.sjoin(
                b[["bid", "geometry"]], subcells_gdf[["subcell_id", "geometry"]], how="inner", predicate="intersects"
            )

            if not joined.empty:
                # Calculate intersection areas per subcell
                building_areas = []
                for subcell_id, group in joined.groupby("subcell_id"):
                    subcell_geom = subcells_gdf.loc[subcells_gdf["subcell_id"] == subcell_id, "geometry"].iloc[0]
                    total_area = 0.0
                    for _, row in group.iterrows():
                        bldg_geom = b.loc[row["bid"], "geometry"]
                        try:
                            intersection = bldg_geom.intersection(subcell_geom)
                            if hasattr(intersection, "area"):
                                total_area += intersection.area
                        except Exception:
                            continue
                    building_areas.append({"subcell_id": subcell_id, "building_area_sqm": total_area})

                areas_df = pd.DataFrame(building_areas)
                subcells_gdf = subcells_gdf.merge(areas_df, on="subcell_id", how="left", suffixes=("", "_new"))
                subcells_gdf["building_area_sqm"] = subcells_gdf["building_area_sqm_new"].fillna(
                    subcells_gdf["building_area_sqm"]
                )
                subcells_gdf = subcells_gdf.drop(columns=["building_area_sqm_new"], errors="ignore")

    # Roads aggregation with sjoin
    if not roads.empty:
        r = roads[roads.geometry.is_valid].copy()
        if not r.empty:
            r = r.reset_index(drop=True)
            r["rid"] = r.index

            joined = gpd.sjoin(
                r[["rid", "geometry"]], subcells_gdf[["subcell_id", "geometry"]], how="inner", predicate="intersects"
            )

            if not joined.empty:
                road_lengths = []
                for subcell_id, group in joined.groupby("subcell_id"):
                    subcell_geom = subcells_gdf.loc[subcells_gdf["subcell_id"] == subcell_id, "geometry"].iloc[0]
                    total_length = 0.0
                    for _, row in group.iterrows():
                        road_geom = r.loc[row["rid"], "geometry"]
                        try:
                            intersection = road_geom.intersection(subcell_geom)
                            if hasattr(intersection, "length"):
                                total_length += intersection.length
                        except Exception:
                            continue
                    road_lengths.append({"subcell_id": subcell_id, "road_length_m": total_length})

                lengths_df = pd.DataFrame(road_lengths)
                subcells_gdf = subcells_gdf.merge(lengths_df, on="subcell_id", how="left", suffixes=("", "_new"))
                subcells_gdf["road_length_m"] = subcells_gdf["road_length_m_new"].fillna(subcells_gdf["road_length_m"])
                subcells_gdf = subcells_gdf.drop(columns=["road_length_m_new"], errors="ignore")

    # 4. Calculate metrics and apply criteria (vectorized)
    print("\n4. Applying qualification criteria...")

    subcells_gdf["area_sqkm"] = subcells_gdf.geometry.area / 1_000_000
    subcells_gdf["road_density_km_per_sqkm"] = np.where(
        subcells_gdf["area_sqkm"] > 0, (subcells_gdf["road_length_m"] / 1000) / subcells_gdf["area_sqkm"], 0
    )

    # Parent population and area lookup
    parent_pop = urban_cells.set_index("cell_id")["population"]
    parent_area = urban_cells.set_index("cell_id").geometry.area

    subcells_gdf["parent_population"] = subcells_gdf["parent_hex_id"].map(parent_pop)
    subcells_gdf["parent_area"] = subcells_gdf["parent_hex_id"].map(parent_area)
    subcells_gdf["sub_area"] = subcells_gdf.geometry.area
    subcells_gdf["pop_share"] = np.where(
        subcells_gdf["parent_area"] > 0, subcells_gdf["sub_area"] / subcells_gdf["parent_area"], 0
    )
    subcells_gdf["est_population"] = subcells_gdf["parent_population"] * subcells_gdf["pop_share"]

    # Vectorized qualification check
    presence_ok = (~require_local_presence) | (
        (subcells_gdf["building_area_sqm"] >= 100)
        | (subcells_gdf["est_population"] >= 0.01 * subcells_gdf["parent_population"])
    )

    subcells_gdf["is_urban"] = presence_ok & (
        (subcells_gdf["building_area_sqm"] >= min_building_area)
        | (subcells_gdf["road_density_km_per_sqkm"] >= min_road_density)
    )

    urban_sub = subcells_gdf[subcells_gdf["is_urban"]].copy()
    print(f"Qualified sub-cells: {len(urban_sub)} of {len(subcells_gdf)}")

    # 5. Build refined set
    print("\n5. Building refined urban area...")

    non_edge = urban_cells[~urban_cells["cell_id"].isin(edge_hexagons)].copy()

    if urban_sub.empty:
        refined = non_edge
    else:
        # Map columns to original schema
        urban_sub = urban_sub.rename(
            columns={"building_area_sqm": "total_building_area_sqm", "road_length_m": "total_road_length_m"}
        )

        # Inherit selected fields with proportional scaling for population
        keep_cols = [c for c in urban_cells.columns if c != "geometry"]
        for c in keep_cols:
            if c not in urban_sub.columns:
                urban_sub[c] = 0

        urban_sub["population"] = urban_sub["est_population"]
        max_id = non_edge["cell_id"].max() if len(non_edge) else 0
        urban_sub["cell_id"] = range(int(max_id) + 1, int(max_id) + 1 + len(urban_sub))

        refined = pd.concat([non_edge, urban_sub[urban_cells.columns]], ignore_index=True)
        refined = gpd.GeoDataFrame(refined, crs=target_crs)

    # 6. Keep largest connected component
    if len(refined) <= 1:
        return refined

    print("\n6. Ensuring connectivity...")
    refined = refined.reset_index(drop=True)
    refined["cell_id"] = range(len(refined))

    # Build adjacency for refined cells
    ref_adjacency = build_adjacency_graph(refined, id_col="cell_id")

    # Use scipy for connected components if available
    try:
        from scipy.sparse import csr_matrix
        from scipy.sparse.csgraph import connected_components

        cell_ids = list(refined["cell_id"])
        id_to_idx = {cid: i for i, cid in enumerate(cell_ids)}
        n = len(cell_ids)

        rows, cols = [], []
        for cid in cell_ids:
            for neighbor in ref_adjacency.get(cid, set()):
                if neighbor in id_to_idx:
                    rows.append(id_to_idx[cid])
                    cols.append(id_to_idx[neighbor])

        if rows:
            adj_matrix = csr_matrix(([1] * len(rows), (rows, cols)), shape=(n, n))
            n_components, labels = connected_components(adj_matrix, directed=False)
        else:
            n_components, labels = len(refined), list(range(len(refined)))

        refined["component"] = labels

        # Keep largest by total area
        comp_areas = refined.groupby("component").apply(lambda g: g.geometry.area.sum())
        best_comp = comp_areas.idxmax()
        refined = refined[refined["component"] == best_comp].copy()

        print(f"Kept largest component ({len(refined)} elements) of {n_components} total")

    except ImportError:
        # Fallback to manual BFS
        visited = set()
        comps = []

        for cid in refined["cell_id"]:
            if cid in visited:
                continue
            q = deque([cid])
            visited.add(cid)
            comp = []
            while q:
                k = q.popleft()
                comp.append(k)
                for neighbor in ref_adjacency.get(k, set()):
                    if neighbor not in visited and neighbor in set(refined["cell_id"]):
                        visited.add(neighbor)
                        q.append(neighbor)
            comps.append(comp)

        if len(comps) > 1:
            best = max(comps, key=lambda c: refined[refined["cell_id"].isin(c)].geometry.area.sum())
            refined = refined[refined["cell_id"].isin(best)].copy()

    refined = refined.drop(columns=["component"], errors="ignore").reset_index(drop=True)
    print(f"Final refined urban area: {len(refined)} elements")

    return refined

In [ ]:
# Apply edge refinement with connectivity check
if not urban_cells.empty:
    print("Applying OPTIMIZED edge refinement with connectivity check...")
    print("This process tightens shoreline artifacts by requiring local presence.")

    # Configure refinement parameters (tightened for Oslo's fjords/bridges)
    SUBDIVISION_FACTOR = 4  # 4x4 = 16 sub-cells per edge hexagon
    MIN_BUILDING_AREA = 600  # m² minimum building area per sub-cell
    MIN_ROAD_DENSITY = 3.0  # km/sq km minimum road density per sub-cell

    refined_urban_cells = refine_urban_edges_optimized(
        urban_cells=urban_cells,
        all_analysis_data=analysis_data,
        buildings=city_buildings,
        roads=city_roads,
        subdivision_factor=SUBDIVISION_FACTOR,
        min_building_area=MIN_BUILDING_AREA,
        min_road_density=MIN_ROAD_DENSITY,
        require_local_presence=True,
    )

    urban_cells = refined_urban_cells
    print(f"Refinement done. Elements: {len(urban_cells)}")
else:
    print("No urban area detected - skipping edge refinement")

In [ ]:
# Create comparison visualization (before/after edge refinement)
if not urban_cells.empty and "refined_urban_cells" in locals():
    print("Creating before/after comparison visualization...")

    # Convert both to WGS84 for visualization
    try:
        # Re-run to get the original urban cells before refinement
        original_cells_viz = run_growing_method(
            analysis_data.copy(),
            urban_strictness=URBAN_STRICTNESS,
            multi_seed=USE_MULTI_SEED,
            min_seed_cluster_size=MIN_SEED_CLUSTER_SIZE,
        )
        original_cells_viz = original_cells_viz.to_crs("EPSG:4326")
        refined_cells_viz = urban_cells.to_crs("EPSG:4326")

        # Create base map with all analysis cells
        all_cells_viz = analysis_data.to_crs("EPSG:4326")
        all_cells_viz["population_display"] = all_cells_viz["population"].round(0).astype(int)

        m_comparison = all_cells_viz.explore(
            color="lightgrey",
            tooltip=["population_display"],
            popup=False,
            style_kwds=dict(fillColor="lightgrey", color="darkgrey", fillOpacity=0.3, weight=1, opacity=0.5),
            name="All Analysis Cells",
        )

        # Add original urban area
        original_cells_viz.explore(
            m=m_comparison,
            color="blue",
            tooltip=False,
            popup=False,
            style_kwds=dict(fillColor="blue", color="darkblue", fillOpacity=0.4, weight=2, opacity=0.8),
            name="Original Urban Area",
        )

        # Add refined urban area
        refined_cells_viz.explore(
            m=m_comparison,
            color="red",
            tooltip=False,
            popup=False,
            style_kwds=dict(fillColor="red", color="darkred", fillOpacity=0.6, weight=1, opacity=0.9),
            name="Refined Urban Area",
        )

        # Add layer control
        folium.LayerControl().add_to(m_comparison)

        print("Comparison map legend:")
        print("- Blue: Original urban area (full hexagons)")
        print("- Red: Refined urban area (with sub-cell precision)")
        print("- Grey: All analysis cells (background)")

        display(m_comparison)

    except Exception as e:
        print(f"Could not create comparison visualization: {e}")
        print("Proceeding with refined results only...")

else:
    print("Comparison visualization not available - proceeding with current results")

In [ ]:
# Save refined urban area results with a hull
if not urban_cells.empty:
    print(f"Saving results for {len(urban_cells)} refined urban elements...")

    # Define output paths
    method = "growing_refined"
    output_dir = Path(DATA_DIR)
    urban_cells_geojson = output_dir / f"urban_cells_{CITY_SHORT_NAME}_{method}.geojson"
    urban_gpkg = output_dir / "urban_areas.gpkg"
    analysis_gpkg = output_dir / "population_density.gpkg"

    cells_layer = f"urban_cells_{CITY_SHORT_NAME}_{method}"
    hull_layer = f"urban_area_{CITY_SHORT_NAME}_{method}"
    analysis_layer = f"population_density_with_buildings_roads_{CITY_SHORT_NAME}"

    # Convert to WGS84 for saving
    urban_save = urban_cells.to_crs("EPSG:4326")

    # Save individual cells
    urban_save.to_file(str(urban_cells_geojson), driver="GeoJSON")
    urban_save.to_file(str(urban_gpkg), layer=cells_layer, driver="GPKG")

    # IMPROVED: Create hole-filled urban area hull
    print("Creating hole-filled urban area hull...")

    try:
        # Ensure geometries are valid before union
        urban_hull_geometries = urban_cells.copy()
        urban_hull_geometries["geometry"] = urban_hull_geometries["geometry"].buffer(0)

        # Create union of all urban geometries
        from shapely.ops import unary_union

        hull_geometry = unary_union(urban_hull_geometries.geometry)

        # Fill holes by applying a negative and then positive buffer
        try:
            # Buffer in by 10 meters to fill small holes
            smoothed_geometry = hull_geometry.buffer(-10)
            # Buffer out by 10 meters to restore size
            smoothed_geometry = smoothed_geometry.buffer(10)
            # Re-assign the smoothed geometry
            hull_geometry = smoothed_geometry
        except Exception as e:
            print(f"Error buffering geometry to fill holes: {e}. Using original hull geometry.")

        # Step 5: Create GeoDataFrame with proper attributes
        urban_hull = gpd.GeoDataFrame(
            {
                "city_name": [CITY_NAME],
                "method": [method],
                "cell_count": [len(urban_cells)],
                "total_population": [urban_cells["population"].sum()],
                "total_building_area_sqm": [urban_cells["total_building_area_sqm"].sum()],
                "area_sqkm": [hull_geometry.area / 1_000_000],
                "holes_filled": [True],
                "geometry": [hull_geometry],
            },
            crs=urban_cells.crs,
        )

        # Convert to WGS84 for saving
        urban_hull_save = urban_hull.to_crs("EPSG:4326")
        urban_hull_save.to_file(str(urban_gpkg), layer=hull_layer, driver="GPKG")

        print("Hole-filled urban hull created successfully:")
        print(f"- Hull area: {urban_hull['area_sqkm'].iloc[0]:.2f} sq km")
        print(f"- Geometry type: {hull_geometry.geom_type}")
        print(f"- Is valid: {hull_geometry.is_valid}")
        print("- Holes filled: Yes")

        if hull_geometry.geom_type == "Polygon":
            print(f"- Interior holes: {len(hull_geometry.interiors)} (should be 0)")
        elif hull_geometry.geom_type == "MultiPolygon":
            total_holes = sum(len(poly.interiors) for poly in hull_geometry.geoms if poly.geom_type == "Polygon")
            print(f"- Total interior holes: {total_holes} (should be 0)")
            print(f"- Number of separate areas: {len(hull_geometry.geoms)}")

    except Exception as e:
        print(f"Error creating hole-filled hull: {e}")

    # Save analysis dataset
    analysis_data.to_file(str(analysis_gpkg), layer=analysis_layer, driver="GPKG")

    print("\nResults saved to:")
    print(f"- Refined urban elements (GeoJSON): {urban_cells_geojson}")
    print(f"- Refined urban elements (GPKG): {urban_gpkg}/{cells_layer}")
    print(f"- Hole-filled urban hull (GPKG): {urban_gpkg}/{hull_layer}")
    print(f"- Analysis data: {analysis_gpkg}/{analysis_layer}")
else:
    print("No urban area to save")

In [ ]:
# Map the urban hull to check

if not urban_cells.empty:
    print("Creating urban hull visualization with overlaid cells...")

    # Create urban hull for visualization
    from shapely.ops import unary_union

    try:
        # Ensure geometries are valid
        urban_viz_cells = urban_cells.copy()
        urban_viz_cells["geometry"] = urban_viz_cells["geometry"].buffer(0)

        # Create hull geometry
        hull_geometry = unary_union(urban_viz_cells.geometry.tolist())
        if not hull_geometry.is_valid:
            hull_geometry = hull_geometry.buffer(0)

        # Create hull GeoDataFrame
        urban_hull_viz = gpd.GeoDataFrame(
            {"name": ["Urban Area Hull"], "area_sqkm": [hull_geometry.area / 1_000_000], "geometry": [hull_geometry]},
            crs=urban_cells.crs,
        )

        # Convert to WGS84 for visualization
        hull_viz_4326 = urban_hull_viz.to_crs("EPSG:4326")
        cells_viz_4326 = urban_cells.to_crs("EPSG:4326")

        # Create base map with urban hull (transparent fill, pink outline)
        m_hull = hull_viz_4326.explore(
            color="hotpink",  # Pink outline
            tooltip=["name", "area_sqkm"],
            popup=True,
            style_kwds=dict(
                fillColor="hotpink",  # Fill color (will be transparent)
                color="hotpink",  # Outline color
                fillOpacity=0.0,  # Completely transparent fill
                weight=3,  # Thick outline
                opacity=1.0,  # Solid outline
            ),
            name="Urban Area Hull",
        )

        # Add urban cells on top with transparency
        cells_viz_4326.explore(
            m=m_hull,  # Add to existing map
            color="blue",
            tooltip=["population", "total_building_area_sqm", "road_density_km_per_sqkm"],
            popup=True,
            style_kwds=dict(
                fillColor="blue",
                color="darkblue",
                fillOpacity=0.4,  # Semi-transparent cells
                weight=1,  # Thin outline
                opacity=0.8,
            ),
            name="Urban Cells",
        )

        # Add layer control
        folium.LayerControl().add_to(m_hull)

        print("Urban hull visualization created:")
        print(f"- Hull area: {hull_viz_4326['area_sqkm'].iloc[0]:.2f} sq km")
        print(f"- Urban cells: {len(cells_viz_4326)} elements")
        print("- Pink outline: Urban area boundary")
        print("- Blue cells: Individual urban elements (semi-transparent)")

        display(m_hull)

    except Exception as e:
        print(f"Error creating hull visualization: {e}")
        print("Displaying urban cells only...")

        # Fallback: just show the urban cells
        cells_viz_4326 = urban_cells.to_crs("EPSG:4326")
        m_fallback = cells_viz_4326.explore(
            color="blue",
            tooltip=["population", "total_building_area_sqm"],
            popup=True,
            style_kwds=dict(fillOpacity=0.6, weight=1),
            name="Urban Cells",
        )
        display(m_fallback)

else:
    print("No urban area to visualize")

## Save Results


In [ ]:
# Save refined urban area results
if not urban_cells.empty:
    print(f"Saving results for {len(urban_cells)} refined urban elements...")

    # Define output paths
    method = "growing_refined"  # Updated to reflect edge refinement
    output_dir = Path(DATA_DIR)
    urban_cells_geojson = output_dir / f"urban_cells_{CITY_SHORT_NAME}_{method}.geojson"
    urban_gpkg = output_dir / "urban_areas.gpkg"
    analysis_gpkg = output_dir / "population_density.gpkg"

    cells_layer = f"urban_cells_{CITY_SHORT_NAME}_{method}"
    hull_layer = f"urban_area_{CITY_SHORT_NAME}_{method}"
    analysis_layer = f"population_density_with_buildings_roads_{CITY_SHORT_NAME}"

    # Convert to WGS84 for saving
    urban_save = urban_cells.to_crs("EPSG:4326")

    # Save individual cells
    urban_save.to_file(str(urban_cells_geojson), driver="GeoJSON")
    urban_save.to_file(str(urban_gpkg), layer=cells_layer, driver="GPKG")

    # Create and save urban area hull
    urban_hull = gpd.GeoDataFrame([urban_cells.unary_union], columns=["geometry"], crs=urban_cells.crs).to_crs(
        "EPSG:4326"
    )

    urban_hull.to_file(str(urban_gpkg), layer=hull_layer, driver="GPKG")

    # Save analysis dataset
    analysis_data.to_file(str(analysis_gpkg), layer=analysis_layer, driver="GPKG")

    print("Results saved to:")
    print(f"- Refined urban elements (GeoJSON): {urban_cells_geojson}")
    print(f"- Refined urban elements (GPKG): {urban_gpkg}/{cells_layer}")
    print(f"- Refined urban hull (GPKG): {urban_gpkg}/{hull_layer}")
    print(f"- Analysis data: {analysis_gpkg}/{analysis_layer}")
else:
    print("No urban area to save")

## Final Summary


In [ ]:
# Print comprehensive summary
print(f"\n{'=' * 60}")
print(f"URBAN AREA ANALYSIS SUMMARY FOR {CITY_NAME.upper()}")
print(f"{'=' * 60}")

print("\nInput Data:")
print(f"- Total hexagons analyzed: {len(analysis_data):,}")
print(f"- Total population: {analysis_data['population'].sum():,}")
print(f"- Total building area: {analysis_data['total_building_area_sqm'].sum():,.0f} m²")
print(f"- Total buildings: {analysis_data['building_count'].sum():,}")

if "road_density_km_per_sqkm" in analysis_data.columns:
    total_roads = analysis_data["total_road_length_m"].sum() / 1000
    avg_density = analysis_data["road_density_km_per_sqkm"].mean()
    print(f"- Total road length: {total_roads:.1f} km")
    print(f"- Average road density: {avg_density:.2f} km/sq km")

if not urban_cells.empty:
    urban_pop_pct = (urban_cells["population"].sum() / analysis_data["population"].sum()) * 100
    urban_area_pct = (len(urban_cells) / len(analysis_data)) * 100
    metrics_used = 3 if "road_density_km_per_sqkm" in urban_cells.columns else 2

    print("\nUrban Area Results (After Edge Refinement):")
    print("- Algorithm: Region growing with hole filling + edge refinement")
    print(
        f"- Metrics used: {metrics_used} (population, building area"
        + (", road density" if metrics_used == 3 else "")
        + ")"
    )
    print(f"- Urban elements: {len(urban_cells):,} (hexagons + refined sub-cells)")
    print(f"- Urban population: {urban_cells['population'].sum():,} ({urban_pop_pct:.1f}% of total)")

    urban_building_area = urban_cells["total_building_area_sqm"].sum()
    urban_building_pct = (urban_building_area / analysis_data["total_building_area_sqm"].sum()) * 100
    print(f"- Urban building area: {urban_building_area:,.0f} m² ({urban_building_pct:.1f}% of total)")

    # Calculate total refined area
    total_urban_area_sqkm = urban_cells.geometry.area.sum() / 1_000_000
    total_analysis_area_sqkm = analysis_data.geometry.area.sum() / 1_000_000
    refined_area_pct = (total_urban_area_sqkm / total_analysis_area_sqkm) * 100
    print(f"- Refined urban area: {total_urban_area_sqkm:.2f} sq km ({refined_area_pct:.1f}% of total area)")

    if "road_density_km_per_sqkm" in urban_cells.columns:
        urban_road_length = urban_cells["total_road_length_m"].sum() / 1000
        urban_road_pct = (urban_road_length / total_roads) * 100
        urban_avg_density = urban_cells["road_density_km_per_sqkm"].mean()
        print(f"- Urban road length: {urban_road_length:.1f} km ({urban_road_pct:.1f}% of total)")
        print(f"- Urban road density: {urban_avg_density:.2f} km/sq km")
else:
    print("\nNo urban area identified")

print(f"\n{'=' * 60}")
print("ANALYSIS COMPLETE")
print(f"{'=' * 60}")